# Molecular Mechanics Core

This notebook shows the molecular-mechanics layer as geometry plus diagnostics: topology, bonded force terms, Coulomb interactions, trajectory snapshots, and per-term energy decomposition.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.diagnostics import summarize_md_result
from mlx_atomistic.examples import bonded_chain_example, charged_dimer_example
from mlx_atomistic.md import SimulationConfig, simulate_nve
from mlx_atomistic.runtime import get_runtime_info

get_runtime_info()

In [ ]:
def set_equal_3d_axes(ax, positions, padding=0.25):
    positions = np.asarray(positions, dtype=float)
    lower = positions.min(axis=0)
    upper = positions.max(axis=0)
    center = 0.5 * (lower + upper)
    radius = 0.5 * np.max(upper - lower) + padding
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def plot_topology_3d(ax, positions, topology=None, charges=None, title=None):
    positions = np.asarray(positions, dtype=float)
    if charges is None:
        colors = "#4f83cc"
        scatter = ax.scatter(positions[:, 0], positions[:, 1], positions[:, 2], s=80, c=colors)
    else:
        charges = np.asarray(charges, dtype=float)
        limit = max(1e-12, np.max(np.abs(charges)))
        scatter = ax.scatter(
            positions[:, 0],
            positions[:, 1],
            positions[:, 2],
            s=100,
            c=charges,
            cmap="coolwarm",
            vmin=-limit,
            vmax=limit,
        )

    if topology is not None:
        for i, j in np.array(topology.bonds, dtype=int):
            segment = positions[[i, j]]
            ax.plot(segment[:, 0], segment[:, 1], segment[:, 2], color="#333333", linewidth=2)

    for atom_index, xyz in enumerate(positions):
        ax.text(xyz[0], xyz[1], xyz[2], f" {atom_index}", fontsize=9)

    if title:
        ax.set_title(title)
    ax.set_xlabel("x / σ")
    ax.set_ylabel("y / σ")
    ax.set_zlabel("z / σ")
    set_equal_3d_axes(ax, positions)
    return scatter

## Bonded Chain

The first example is a four-particle chain with bond, angle, and dihedral terms. The plot shows atom indices and bonded connectivity.

In [ ]:
positions, velocities, topology, force_terms = bonded_chain_example()

{
    "positions_shape": np.array(positions).shape,
    "bonds": np.array(topology.bonds).tolist(),
    "angles": np.array(topology.angles).tolist(),
    "dihedrals": np.array(topology.dihedrals).tolist(),
    "force_terms": [term.name for term in force_terms],
}

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
plot_topology_3d(ax, positions, topology=topology, title="Initial bonded chain")
plt.tight_layout()

In [ ]:
result = simulate_nve(
    positions,
    velocities,
    force_terms=force_terms,
    config=SimulationConfig(dt=0.0005, steps=200, sample_interval=10),
)

summarize_md_result(result)

In [ ]:
frames = np.array(result.sampled_positions)
frame_ids = [0, len(frames) // 2, len(frames) - 1]

fig = plt.figure(figsize=(12, 4))
for panel, frame_id in enumerate(frame_ids, start=1):
    ax = fig.add_subplot(1, 3, panel, projection="3d")
    step = int(np.array(result.sampled_steps)[frame_id])
    plot_topology_3d(ax, frames[frame_id], topology=topology, title=f"step {step}")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, series in result.potential_energy_by_term.items():
    axes[0].plot(np.array(series), label=name)
axes[0].plot(np.array(result.potential_energy), label="total", color="black", linewidth=1)
axes[0].set_xlabel("step")
axes[0].set_ylabel("potential energy / ε")
axes[0].set_title("Per-term potential energy")
axes[0].legend()

axes[1].plot(np.array(result.energy_drift))
axes[1].set_xlabel("step")
axes[1].set_ylabel("ΔE / ε")
axes[1].set_title("NVE total-energy drift")

plt.tight_layout()

## Charged Dimer

The second example isolates direct Coulomb behavior. The color map shows partial charge sign and magnitude.

In [ ]:
charged_positions, charged_velocities, charged_topology, charged_terms = charged_dimer_example()
charges = np.array(charged_topology.partial_charges)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
scatter = plot_topology_3d(
    ax,
    charged_positions,
    topology=charged_topology,
    charges=charges,
    title="Charged dimer",
)
fig.colorbar(scatter, ax=ax, shrink=0.65, label="partial charge")
plt.tight_layout()

In [ ]:
charged = simulate_nve(
    charged_positions,
    charged_velocities,
    force_terms=charged_terms,
    config=SimulationConfig(dt=0.0005, steps=20, sample_interval=5),
)

summarize_md_result(charged)

In [ ]:
charged_frames = np.array(charged.sampled_positions)
charged_steps = np.array(charged.sampled_steps)
distances = np.linalg.norm(charged_frames[:, 1] - charged_frames[:, 0], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(np.array(charged.potential_energy_by_term["coulomb"]), label="coulomb")
axes[0].set_xlabel("step")
axes[0].set_ylabel("potential energy / ε")
axes[0].set_title("Coulomb energy")

axes[1].plot(charged_steps, distances, marker="o")
axes[1].set_xlabel("sampled step")
axes[1].set_ylabel("distance / σ")
axes[1].set_title("Sampled pair distance")

plt.tight_layout()